In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import time

In [ ]:
# User Agent
headers = {
    'authority': 'www.99acres.com',
    'method': 'GET',
    'path': '/flats-in-Gurgaon-ffid-page-6',
    'scheme': 'https',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-encoding': 'gzip, deflate, br, zstd',
    'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8',
    'cookie': '99_ab=45; GOOGLE_SEARCH_ID=5282341758295235153; ... (trimmed for security) ...',
    'priority': 'u=0, i',
    'sec-ch-ua': '"Chromium";v="140", "Not=A?Brand";v="24", "Google Chrome";v="140"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'none',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36',
}


In [ ]:
City = "Gurgaon"

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import time

City = "Gurgaon"

start = int(input("Enter page number where you got error in last run.\nEnter page number to start from:"))  # Starting Page

# End Page number- you can change is for start i am taking 10pages at a time,
# as IPs are gettig block after some time
end = start + 10

pageNumber = start
req = 0

flats = pd.DataFrame()

while pageNumber < end:
    i = 1
    url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
    page = requests.get(url, headers=headers)
    pageSoup = BeautifulSoup(page.content, 'html.parser')
    req += 1

    try:
        for soup in pageSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):

            # Extract property name and property sub-name
            try:
                property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
                # Extract link
                link = soup.select_one('a.srpTuple__propertyName')['href']
                society = soup.select_one('#srp_tuple_society_heading').text.strip()
            except:
                continue
            # Detail Page
            page = requests.get(link, headers=headers)
            dpageSoup = BeautifulSoup(page.content, 'html.parser')
            req += 1
            try:
                # price Range
                price = dpageSoup.select_one('#pdPrice2').text.strip()
            except:
                price = ''

            # Area
            try:
                area = soup.select_one('#srp_tuple_price_per_unit_area').text.strip()
            except:
                area = ''
            # Area with Type
            try:
                areaWithType = dpageSoup.select_one('#factArea').text.strip()
            except:
                areaWithType = ''


            # Configuration
            try:
                bedRoom = dpageSoup.select_one('#bedRoomNum').text.strip()
            except:
                bedRoom = ''
            try:
                bathroom = dpageSoup.select_one('#bathroomNum').text.strip()
            except:
                bathroom = ''
            try:
                balcony = dpageSoup.select_one('#balconyNum').text.strip()
            except:
                balcony = ''

            try:
                additionalRoom = dpageSoup.select_one('#additionalRooms').text.strip()
            except:
                additionalRoom = ''


            # Address

            try:
                address = dpageSoup.select_one('#address').text.strip()
            except:
                address = ''
            # Floor Number
            try:
                floorNum = dpageSoup.select_one('#floorNumLabel').text.strip()
            except:
                floorNum = ''

            try:
                facing = dpageSoup.select_one('#facingLabel').text.strip()
            except:
                facing = ''

            try:
                agePossession = dpageSoup.select_one('#agePossessionLbl').text.strip()
            except:
                agePossession = ''

            # Nearby Locations

            try:
                nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
            except:
                nearbyLocations = ''

            # Descriptions
            try:
                description = dpageSoup.select_one('#description').text.strip()
            except:
                description = ''

            # Furnish Details
            try:
                furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#FurnishDetails').select('li')]
            except:
                furnishDetails = ''

            # Features
            if furnishDetails:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[1].select('li')]
                except:
                    features = ''
            else:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
                except:
                    features = ''



            # Rating by Features
            try:
                rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
            except:
                rating = ''
            # print(top_f)

            try:
                # Property ID
                property_id = dpageSoup.select_one('#Prop_Id').text.strip()
            except:
                property_id = ''

            # Create a dictionary with the given variables
            property_data = {
                'property_name': property_name,
                'link': link,
                'society': society,
                'price': price,
                'area': area,
                'areaWithType': areaWithType,
                'bedRoom': bedRoom,
                'bathroom': bathroom,
                'balcony': balcony,
                'additionalRoom': additionalRoom,
                'address': address,
                'floorNum': floorNum,
                'facing': facing,
                'agePossession': agePossession,
                'nearbyLocations': nearbyLocations,
                'description': description,
                'furnishDetails': furnishDetails,
                'features': features,
                'rating': rating,
                'property_id': property_id
            }


            temp_df = pd.DataFrame.from_records([property_data])
            # print(temp_df)
            flats = pd.concat([flats, temp_df], ignore_index=True)
            i += 1
            # if os.path.isfile(csv_file):
            # # Append DataFrame to the existing file without header
            #     temp_df.to_csv(csv_file, mode='a', header=False, index=False)
            # else:
            #     # Write DataFrame to the file with header
            #     temp_df.to_csv(csv_file, mode='a', header=True, index=False)

            if req % 4 == 0:
                time.sleep(10)
            if req % 15 == 0:
                time.sleep(50)
        print(f'{pageNumber} -> {i}')
        pageNumber += 1

    except AttributeError as e:
        print(e)
        print("----------------")
        print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n
              You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")
        csv_file_path = f"flats_{City}_data-page-{start}-{pageNumber-1}.csv"

        # This file will be new every time if start page will chnage, but still taking here mode as append
        if os.path.isfile(csv_file_path):
        # Append DataFrame to the existing file without header
            flats.to_csv(csv_file_path, mode='a', header=False, index=False)
        else:
            # Write DataFrame to the file with header - first time write
            flats.to_csv(csv_file_path, mode='a', header=True, index=False)

Enter page number where you got error in last run.
Enter page number to start from:1


In [ ]:
pageNumber =3
url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
temp = requests.get(url, headers=headers)
tempSoup = BeautifulSoup(temp.content, 'html.parser')

In [ ]:
content = tempSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]')
content[0]

<section data-hydration-on-demand="true"><div class="PseudoTupleRevamp__outerTupleWrap" data-custominfo='{"payload":{"search_results":{"selected_entity_tuples":[{"id":"422937","rank":53,"res_com":"R","attribute":"PSEUDO_GROUPED","attributes":{"rtov_enabled":"Y","rtov_booked":"","isPreLeased":""}}]}}}' data-label="GROUPED_PROJECT_TUPLE.1" data-propid="422937" data-rank="1_1" id="422937" topmost="true"><div class="PseudoTupleRevamp__tupleWrapProject undefined"><div class="PseudoTupleRevamp__projectInnerCount"><div class="tupleNew__imgWrap tupleNew__premiumProject" id="tuple_srp_0_5ljE1hkVNQU"><div class="tupleNew__topLeftItems"><div style="display:flex"><div><img alt="Rera-Tag" class="ImgItem__tag" src="https://static.99acres.com/universalapp/img/rera.png" style="width:52px;height:20px"/><img alt="No-Brokerage-Tag" class="ImgItem__tag" src="https://static.99acres.com/universalapp/img/NBZ.png" style="width:101px;height:20px"/><img alt="3D-Tag" class="ImgItem__tag" src="https://static.99ac

In [ ]:
for soup in tempSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):
  print(soup.select_one('a.srpTuple__propertyName'))

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [ ]:
df = pd.read_csv('/content/flats - flats.csv')

In [ ]:
df.head(1)

,property_name,link,society,price,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating,property_id
0,2 BHK Flat in Krishna Colony,https://www.99acres.com/2-bhk-bedroom-apartmen...,maa bhagwati residency,45 Lac,"₹ 5,000/sq.ft.",Carpet area: 900 (83.61 sq.m.),2 Bedrooms,2 Bathrooms,1 Balcony,NaN,"Krishna Colony, Gurgaon, Haryana",4th of 4 Floors,West,1 to 5 Year Old,"['Chintapurni Mandir', 'State bank ATM', 'Pear...",So with lift.Maa bhagwati residency is one of ...,"['3 Fan', '4 Light', '1 Wardrobe', 'No AC', 'N...","['Feng Shui / Vaastu Compliant', 'Security / F...","['Environment4 out of 5', 'Safety4 out of 5', ...",C68850746
